In [1]:
%pip install -U -r /Users/giahuy/Documents/GitHub/topicmodeling/requirements.txt

  Using cached langchain-1.0.2-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.0.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_google_genai-3.0.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-1.0.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached langgraph-1.0.1-py3-none-any.whl.metadata (7.4 kB)
  Using cached langchain_classic-1.0.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_ai_generativelanguage-0.9.0-py3-none-any.whl.metadata (10 kB)
INFO: pip is looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
  Using cached google_generativeai-0.8.5-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_generativeai-0.8.4-py3-none-any.whl.metadata (4.2 kB)
  Using cached google_generativeai-0.8.3-py3-none-any.whl.metadata (3.9 kB)
  Using cached google_generat

In [2]:
import getpass
import os

try:
    # load environment variables from .env file (requires `python-dotenv`)
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

os.environ["LANGSMITH_TRACING"] = "true"
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
        prompt="Enter your LangSmith API key (optional): "
    )
if "LANGSMITH_PROJECT" not in os.environ:
    os.environ["LANGSMITH_PROJECT"] = getpass.getpass(
        prompt='Enter your LangSmith Project Name (default = "default"): '
    )
    if not os.environ.get("LANGSMITH_PROJECT"):
        os.environ["LANGSMITH_PROJECT"] = "default"

In [11]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

E0000 00:00:1761125519.803683 11961284 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Vietnamese expert in sentiment analysis. Answer concisely and give no explanation."),
    ("user", """Classifying the given text into one of the emotion categories: happiness, sadness, fear, anger, disgust, surprise, love, guilt, shame, pride, jealousy, relief, and hope.
{documents}
""")
])

chain = prompt | model
result = chain.invoke({"documents": "Tôi không hài lòng với quyết định đó"})
print(result)

content='Tức giận' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--6a133b03-fc3f-4281-a950-489e10d67a84-0' usage_metadata={'input_tokens': 66, 'output_tokens': 4, 'total_tokens': 517, 'input_token_details': {'cache_read': 0}}


In [13]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {
        "text": "con bé vẫn hoang mang lắm",
        "emotion": "sadness"
    },
    {
        "text": "tiền làm ra được thì phải hưởng thụ thôi",
        "emotion": "pride"
    },
    {
        "text": "trường bé như lỗ mũi thi vấn đáp all môn",
        "emotion": "anger"
    },
    {
        "text": "ủng hộ em học nhé",
        "emotion": "love"
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Text: {text}"),
    ("ai", "Emotion: {emotion}")
])

few_shot = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Vietnamese expert in sentiment analysis. Classify the given text into one of the emotion categories: happiness, sadness, fear, anger, disgust, surprise, love, guilt, shame, pride, jealousy, relief, and hope. Answer with only the emotion category."),
    few_shot,
    ("human", "Text: {text}")
])

few_shot_chain = few_shot_prompt | model
result = few_shot_chain.invoke({"text": "trường team tỉnh thành team top toàn quốc top toàn quốc ngoái nhé"})
print(result)


content='Emotion: pride' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--46beb1c3-8b1c-40c0-9f5c-7e12c4c751ca-0' usage_metadata={'input_tokens': 131, 'output_tokens': 3, 'total_tokens': 134, 'input_token_details': {'cache_read': 0}}


In [ ]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train_set.csv', 'validation': 'val_set.csv', 'test': 'test_set.csv'}
df = pd.read_csv("hf://datasets/hung20gg/NEU-ESC/" + splits["train"])

In [ ]:
df.head()

,text,sentiment,classification
0,em xin chào các anh chị em em là sinh viên vừa...,0.0,2.0
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,0.0,2.0
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,2.0,3.0
3,chào thầy đăng ký học ghép . môn học hiện hệ t...,0.0,2.0
4,con bé vẫn hoang mang lắm .,2.0,3.0
